# DESCRIPTIVES — Sub-3 h Berlin Marathon Pacing Analysis

**Objective:** Descriptive statistics comparing runners who narrowly broke 3 h (Success: 2:58:00–2:59:59) with those who started at the same half-marathon pace but faded beyond 3:05:00 (Failure).

- `cohort_success.csv` — 7 710 runners
- `cohort_failure.csv` — 1 875 runners

In [1]:
import pathlib, warnings
import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings("ignore")

DATA = pathlib.Path("../data")

success = pd.read_csv(DATA / "cohort_success.csv")
failure = pd.read_csv(DATA / "cohort_failure.csv")

success["outcome"] = "Success"
failure["outcome"] = "Failure"

df = pd.concat([success, failure], ignore_index=True)

PACE_COLS = [
    "pace_05k", "pace_510k", "pace_1015k", "pace_1520k",
    "pace_2025k", "pace_2530k", "pace_3035k", "pace_3540k", "pace_40End",
]

def sec_to_hms(s):
    """Convert seconds to H:MM:SS string."""
    h = int(s // 3600)
    m = int((s % 3600) // 60)
    sec = int(s % 60)
    return f"{h}:{m:02d}:{sec:02d}"

print(f"Combined dataset: {len(df):,} rows")
print(f"  Success: {len(success):,}")
print(f"  Failure: {len(failure):,}")
print(f"\nColumns: {list(df.columns)}")

Combined dataset: 9,585 rows
  Success: 7,710
  Failure: 1,875

Columns: ['year', 'gender', 'name', 'country', 'starting_num', 'age_group', '5km_sec', '10km_sec', '15km_sec', '20km_sec', 'half_sec', '25km_sec', '30km_sec', '35km_sec', '40km_sec', 'net_time_sec', 'pace_05k', 'pace_510k', 'pace_1015k', 'pace_1520k', 'pace_2025k', 'pace_2530k', 'pace_3035k', 'pace_3540k', 'pace_40End', 'pacing_cv', 'outcome']


## Table 1 — Participant Characteristics

In [2]:
import pathlib as _pl

TABLES = _pl.Path("../manuscript/tables")
TABLES.mkdir(parents=True, exist_ok=True)

# --- helpers ---
def mean_sd_hms(series):
    """Return 'H:MM:SS +/- H:MM:SS' for a Series in seconds."""
    return f"{sec_to_hms(series.mean())} \u00b1 {sec_to_hms(series.std())}"

def mean_sd(series, decimals=2):
    return f"{series.mean():.{decimals}f} \u00b1 {series.std():.{decimals}f}"

def rank_biserial(u, n1, n2):
    """Rank-biserial correlation from Mann-Whitney U."""
    return 1 - (2 * u) / (n1 * n2)

def fmt_p(p):
    return "<0.001" if p < 0.001 else f"{p:.3f}"

# --- split groups ---
g_s = df[df.outcome == "Success"]
g_f = df[df.outcome == "Failure"]
n_s, n_f = len(g_s), len(g_f)

# --- N and gender ---
def gender_str(g):
    m = (g.gender == "M").sum()
    f = (g.gender == "F").sum()
    return f"{m} M ({100*m/len(g):.1f}%), {f} F ({100*f/len(g):.1f}%)"

# --- age group top-3 ---
def top_age(g, n=3):
    counts = g.age_group.value_counts().head(n)
    parts = [f"{ag}: {c} ({100*c/len(g):.1f}%)" for ag, c in counts.items()]
    return "; ".join(parts)

# --- build rows ---
rows = []

# N
rows.append(("N", str(n_s), str(n_f), "", ""))

# Gender
rows.append(("Gender", gender_str(g_s), gender_str(g_f), "", ""))

# Age groups
rows.append(("Age group (top 3)", top_age(g_s), top_age(g_f), "", ""))

# Continuous variables
cont_vars = [
    ("Half-marathon time", "half_sec", True),
    ("Finish time", "net_time_sec", True),
    ("Pacing CV (%)", "pacing_cv", False),
]

for label, col, is_time in cont_vars:
    s_vals = g_s[col].dropna()
    f_vals = g_f[col].dropna()
    u_stat, p_val = stats.mannwhitneyu(s_vals, f_vals, alternative="two-sided")
    r = rank_biserial(u_stat, len(s_vals), len(f_vals))
    if is_time:
        s_str = mean_sd_hms(s_vals)
        f_str = mean_sd_hms(f_vals)
    else:
        s_str = mean_sd(s_vals)
        f_str = mean_sd(f_vals)
    rows.append((label, s_str, f_str, fmt_p(p_val), f"{r:.3f}"))

# --- markdown table ---
header = "| Variable | Success (n={:,}) | Failure (n={:,}) | p-value | r (rank-biserial) |".format(n_s, n_f)
sep    = "|---|---|---|---|---|"
lines  = [header, sep]
for r in rows:
    lines.append("| {} | {} | {} | {} | {} |".format(*r))

table1_md = "\n".join(lines)

(TABLES / "Table1.md").write_text(table1_md + "\n", encoding="utf-8")
print("Table 1 saved to ../manuscript/tables/Table1.md\n")
print(table1_md)

Table 1 saved to ../manuscript/tables/Table1.md

| Variable | Success (n=7,710) | Failure (n=1,875) | p-value | r (rank-biserial) |
|---|---|---|---|---|
| N | 7710 | 1875 |  |  |
| Gender | 7287 M (94.5%), 423 F (5.5%) | 1829 M (97.5%), 46 F (2.5%) |  |  |
| Age group (top 3) | 35: 1704 (22.1%); 40: 1683 (21.8%); 30: 1456 (18.9%) | 35: 457 (24.4%); 40: 395 (21.1%); 30: 326 (17.4%) |  |  |
| Half-marathon time | 1:28:05 ± 0:02:01 | 1:28:08 ± 0:00:17 | <0.001 | -0.221 |
| Finish time | 2:58:59 ± 0:00:33 | 3:12:34 ± 0:07:59 | <0.001 | 1.000 |
| Pacing CV (%) | 3.02 ± 2.75 | 11.52 ± 5.05 | <0.001 | 0.929 |


## Table 2 — Segment Paces by Group

In [3]:
SEGMENT_LABELS = [
    "0–5 km", "5–10 km", "10–15 km", "15–20 km",
    "20–25 km", "25–30 km", "30–35 km", "35–40 km", "40–42.2 km",
]

alpha_bonf = 0.05 / len(PACE_COLS)

header = "| Segment | Success (min/km) | Failure (min/km) | p-value | Sig. | r (rank-biserial) |"
sep    = "|---|---|---|---|---|---|"
lines  = [header, sep]

sig_segments = []

for label, col in zip(SEGMENT_LABELS, PACE_COLS):
    s_vals = g_s[col].dropna()
    f_vals = g_f[col].dropna()
    u_stat, p_val = stats.mannwhitneyu(s_vals, f_vals, alternative="two-sided")
    r = rank_biserial(u_stat, len(s_vals), len(f_vals))
    sig = "***" if p_val < alpha_bonf else ""
    if sig:
        sig_segments.append(label)
    s_str = mean_sd(s_vals)
    f_str = mean_sd(f_vals)
    lines.append(f"| {label} | {s_str} | {f_str} | {fmt_p(p_val)} | {sig} | {r:.3f} |")

lines.append("")
lines.append(f"*Bonferroni-corrected threshold: p < {alpha_bonf:.4f} (0.05 / {len(PACE_COLS)})*")

table2_md = "\n".join(lines)

(TABLES / "Table2.md").write_text(table2_md + "\n", encoding="utf-8")
print("Table 2 saved to ../manuscript/tables/Table2.md\n")
print(table2_md)

print(f"\n--- Segments with significant divergence (Bonferroni-corrected) ---")
if sig_segments:
    for s in sig_segments:
        print(f"  * {s}")
else:
    print("  None")

Table 2 saved to ../manuscript/tables/Table2.md

| Segment | Success (min/km) | Failure (min/km) | p-value | Sig. | r (rank-biserial) |
|---|---|---|---|---|---|
| 0–5 km | 4.18 ± 0.16 | 4.14 ± 0.09 | <0.001 | *** | -0.292 |
| 5–10 km | 4.17 ± 0.11 | 4.15 ± 0.05 | <0.001 | *** | -0.275 |
| 10–15 km | 4.17 ± 0.10 | 4.17 ± 0.05 | <0.001 | *** | -0.132 |
| 15–20 km | 4.18 ± 0.09 | 4.23 ± 0.07 | <0.001 | *** | 0.253 |
| 20–25 km | 4.21 ± 0.08 | 4.38 ± 0.25 | <0.001 | *** | 0.664 |
| 25–30 km | 4.26 ± 0.11 | 4.64 ± 0.38 | <0.001 | *** | 0.865 |
| 30–35 km | 4.30 ± 0.14 | 5.00 ± 0.54 | <0.001 | *** | 0.948 |
| 35–40 km | 4.43 ± 0.23 | 5.49 ± 0.70 | <0.001 | *** | 0.966 |
| 40–42.2 km | 4.33 ± 0.27 | 5.29 ± 0.68 | <0.001 | *** | 0.931 |

*Bonferroni-corrected threshold: p < 0.0056 (0.05 / 9)*

--- Segments with significant divergence (Bonferroni-corrected) ---
  * 0–5 km
  * 5–10 km
  * 10–15 km
  * 15–20 km
  * 20–25 km
  * 25–30 km
  * 30–35 km
  * 35–40 km
  * 40–42.2 km


## Key Takeaways

In [4]:
# --- Key Takeaways ---
print("=" * 60)
print("KEY TAKEAWAYS")
print("=" * 60)

# 1. Sample composition
print(f"\n1. SAMPLE: {n_s:,} Success vs {n_f:,} Failure runners "
      f"(ratio {n_s/n_f:.1f}:1)")

# 2. Finish time gap
s_finish = g_s.net_time_sec.mean()
f_finish = g_f.net_time_sec.mean()
gap = f_finish - s_finish
print(f"\n2. FINISH TIME GAP: Failure runners finish ~{sec_to_hms(gap)} slower "
      f"on average ({sec_to_hms(s_finish)} vs {sec_to_hms(f_finish)})")

# 3. HM similarity
s_hm = g_s.half_sec.mean()
f_hm = g_f.half_sec.mean()
hm_diff = abs(f_hm - s_hm)
print(f"\n3. HALF-MARATHON PACE: Groups start with similar HM times "
      f"(diff = {hm_diff:.0f} sec = {sec_to_hms(hm_diff)}), "
      f"confirming cohort matching")

# 4. Pacing variability
s_cv = g_s.pacing_cv.mean()
f_cv = g_f.pacing_cv.mean()
print(f"\n4. PACING VARIABILITY: Failure group has higher CV "
      f"({f_cv:.2f}% vs {s_cv:.2f}%), indicating less even pacing")

# 5. Segment divergence
print(f"\n5. SEGMENT DIVERGENCE: {len(sig_segments)} of {len(PACE_COLS)} segments "
      f"show significant pace differences after Bonferroni correction:")
if sig_segments:
    for s in sig_segments:
        print(f"   - {s}")

# 6. Late-race fade
late_cols = ["pace_3035k", "pace_3540k", "pace_40End"]
late_labels = ["30-35 km", "35-40 km", "40-42.2 km"]
print(f"\n6. LATE-RACE FADE (last 3 segments):")
for lbl, col in zip(late_labels, late_cols):
    s_mean = g_s[col].mean()
    f_mean = g_f[col].mean()
    diff = f_mean - s_mean
    print(f"   {lbl}: Failure +{diff:.2f} min/km slower "
          f"({f_mean:.2f} vs {s_mean:.2f})")

print("\n" + "=" * 60)

KEY TAKEAWAYS

1. SAMPLE: 7,710 Success vs 1,875 Failure runners (ratio 4.1:1)

2. FINISH TIME GAP: Failure runners finish ~0:13:35 slower on average (2:58:59 vs 3:12:34)

3. HALF-MARATHON PACE: Groups start with similar HM times (diff = 3 sec = 0:00:02), confirming cohort matching

4. PACING VARIABILITY: Failure group has higher CV (11.52% vs 3.02%), indicating less even pacing

5. SEGMENT DIVERGENCE: 9 of 9 segments show significant pace differences after Bonferroni correction:
   - 0–5 km
   - 5–10 km
   - 10–15 km
   - 15–20 km
   - 20–25 km
   - 25–30 km
   - 30–35 km
   - 35–40 km
   - 40–42.2 km

6. LATE-RACE FADE (last 3 segments):
   30-35 km: Failure +0.70 min/km slower (5.00 vs 4.30)
   35-40 km: Failure +1.06 min/km slower (5.49 vs 4.43)
   40-42.2 km: Failure +0.96 min/km slower (5.29 vs 4.33)

